In [1]:
from model import *
from tokenizer import *
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

In [2]:
def collate_fn(batch, tokenizer):
    src_texts = [pair[0] for pair in batch]
    tgt_texts = [pair[1] for pair in batch]

    src_tokens = [torch.tensor(tokenizer.encode(text)) for text in src_texts]
    tgt_tokens = [torch.tensor(tokenizer.encode(text)) for text in tgt_texts]
   
    src_padded = pad_sequence(
        src_tokens, 
        batch_first=True, 
        padding_value=tokenizer.pad_id
    )
    tgt_padded = pad_sequence(
        tgt_tokens, 
        batch_first=True, 
        padding_value=tokenizer.pad_id
    )
    
    # Маски для attention (1 = реальный токен, 0 = padding)
    src_mask = (src_padded != tokenizer.pad_id).unsqueeze(1).unsqueeze(2).long()
    tgt_mask = (tgt_padded != tokenizer.pad_id).unsqueeze(1).unsqueeze(2).long()
    
    # Causal mask для декодера (чтобы не видеть будущие токены)
    seq_len = tgt_padded.size(1)
    subsequent_mask = torch.tril(torch.ones(1, seq_len, seq_len)).type_as(tgt_mask)
    tgt_mask = tgt_mask & subsequent_mask
    
    return src_padded, tgt_padded, src_mask, tgt_mask

In [ ]:
dataset = pd.read_csv('../data/random_words.csv')

In [4]:
dataset.head()

,word,rev_word
0,bVrp,prVb
1,iVgRVIfL,LfIVRgVi
2,cbfnoGMbJmTPSI,ISPTmJbMGonfbc
3,oCLrZaWZkSBvrj,jrvBSkZWaZrLCo
4,Wvgfygw,wgyfgvW


In [ ]:
def is_valid_series(s):
    return s.notna() & (s != '')

cleaned_dataset = dataset[
    is_valid_series(dataset['word']) & 
    is_valid_series(dataset['rev_word'])
]
print(f"Оригинал: {len(dataset)} → Очищено: {len(cleaned_dataset)}")

Оригинал: 100000 → Очищено: 99993


In [6]:
dataset = cleaned_dataset

In [7]:
all_chars = set(''.join(dataset['word'].dropna()))
tokenizer = CharTokenizer(''.join(sorted(all_chars)))

In [8]:
model = MiniTransformer(
    vocab_size = tokenizer.vocab_size,
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=512
)

In [10]:
from sklearn.model_selection import train_test_split

if isinstance(dataset, pd.DataFrame):
    dataset_list = [(row[0], row[1]) for _, row in dataset.iterrows()]
else:
    dataset_list = list(dataset)

train_data, val_data = train_test_split(dataset_list, test_size=0.1)

train_data = train_data.tolist() if hasattr(train_data, 'tolist') else list(train_data)
val_data = val_data.tolist() if hasattr(val_data, 'tolist') else list(val_data)

C:\Users\stepa_aaa\AppData\Local\Temp\ipykernel_10248\103554657.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dataset_list = [(row[0], row[1]) for _, row in dataset.iterrows()]


In [11]:
num_epochs = 50
batch_size = 64
best_val_loss = float('inf')
patience = 5
no_improve_count = 0

In [12]:
train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True,
    collate_fn=lambda batch: collate_fn(batch, tokenizer)  # ← именно так!
)

val_loader = DataLoader(
    val_data,
    batch_size=64,
    collate_fn=lambda batch: collate_fn(batch, tokenizer)
)

In [17]:
import torch
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]', leave=False)
    
    for src, tgt, src_mask, tgt_mask in train_loop:
        src, tgt = src.to(device), tgt.to(device)
        src_mask, tgt_mask = src_mask.to(device), tgt_mask.to(device)

        optimizer.zero_grad()
        
        output = model(
            src,
            tgt[:, :-1],
            src_mask,
            tgt_mask[:, :, :-1, :-1]
        )
        loss = criterion(output.view(-1, tokenizer.vocab_size), tgt[:, 1:].reshape(-1))

        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
        train_loop.set_postfix({'Train Loss': loss.item()})
    
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    val_loop = tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]', leave=False)
    
    with torch.no_grad():
        for src, tgt, src_mask, tgt_mask in val_loop:
            src, tgt = src.to(device), tgt.to(device)
            src_mask, tgt_mask = src_mask.to(device), tgt_mask.to(device)

            output = model(src, tgt[:, :-1], src_mask, tgt_mask[:, :, :-1, :-1])
            loss = criterion(output.view(-1, tokenizer.vocab_size), tgt[:, 1:].reshape(-1))
            val_loss += loss.item()
            
            val_loop.set_postfix({'Val Loss': loss.item()})

    val_loss /= len(val_loader)

    print(f'Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve_count = 0
        torch.save(model.state_dict(), '../models/best_model.pth')
        print('Model saved!')
    else:
        no_improve_count += 1
        if no_improve_count >= patience:
            print(f'Early stopping triggered at epoch {epoch+1}!')
            break

Epoch 1, Train Loss: 2.1820, Val Loss: 1.4846
Model saved!


Epoch 2, Train Loss: 1.2506, Val Loss: 0.6170
Model saved!


Epoch 3, Train Loss: 0.7346, Val Loss: 0.3932
Model saved!


Epoch 4, Train Loss: 0.5053, Val Loss: 0.2841
Model saved!


Epoch 5, Train Loss: 0.3718, Val Loss: 0.1394
Model saved!


...

Epoch 22, Train Loss: 0.0512, Val Loss: 0.0112


Epoch 23, Train Loss: 0.0513, Val Loss: 0.0251


Epoch 24, Train Loss: 0.0453, Val Loss: 0.0071
Model saved!


Epoch 25, Train Loss: 0.0462, Val Loss: 0.0138


KeyboardInterrupt: 